In [1]:
# ============================================================
# Cell 1: Imports
# ============================================================
import os, time, requests
import pandas as pd
from datetime import datetime
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)


In [3]:
# ============================================================
# Cell 2: Configuration (only this cell needs to be updated)
# ============================================================
# QA   : https://findata-qa.worldbank.org:9047
# DEV  : https://findata-dev.worldbank.org:9047
# PROD : https://findata-prod.worldbank.org:9047   <- confirm exact host with IT
BASE_URL = "https://findata-prod.worldbank.org:9047".rstrip("/")

# Token is specific to each environment. It is asked for at run time.
TOKEN    = "tTQlvmAkRWKs8Kc5pOy0XFU+yvAPwsMlSwy9ZVduLyk6k6FhAhE22k4GT7wqHQ==" 

# Confirm the schema for the target environment with IT
SCHEMA = ["publishFINDATAREP", "IFCTRE", "Operational_Data", "Common"]

BUSINESS_DATE = "2026-08-31"            # date to extract (YYYY-MM-DD)
L3_FILTER   = []
# L3_FILTER     = ["LAM", "LAM_LIQUIDTY", "LAM_STRATEGIC"]   # set to [] for no filter

# Output folder (keep only one active line)
# OUTPUT_DIR = r"C:\Users\mkeita9\OneDrive - WBG\1. Work Documents\4. Projects & Initiatives\Murex_Testing_LAM_Cash\08_DR\DR4\01.Inputs\1_Datamart_Extracts\4_Murex_PL_Summary"
OUTPUT_DIR = r"C:\Users\mkeita9\OneDrive - WBG\1. Work Documents\14. Audits\2026"

HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
SCHEMA_PATH = ".".join(SCHEMA)


In [4]:
def run_query(sql, poll=1.0, page=500):
    """Run SQL on Dremio and return a DataFrame."""
    r = requests.post(f"{BASE_URL}/api/v3/sql", headers=HEADERS, json={"sql": sql}, verify=True)
    r.raise_for_status()
    job_id = r.json()["id"]

    while True:
        s = requests.get(f"{BASE_URL}/api/v3/job/{job_id}", headers=HEADERS, verify=True)
        s.raise_for_status()
        info = s.json()
        if info["jobState"] == "COMPLETED":
            break
        if info["jobState"] in ("FAILED", "CANCELED"):
            raise RuntimeError(f"Job {info['jobState']}: {info.get('errorMessage')}")
        time.sleep(poll)

    rows, columns, offset = [], None, 0
    while True:
        res = requests.get(f"{BASE_URL}/api/v3/job/{job_id}/results",
                           headers=HEADERS, params={"offset": offset, "limit": page}, verify=True)
        res.raise_for_status()
        data = res.json()
        if columns is None:
            columns = [c["name"] for c in data.get("schema", [])]
        batch = data.get("rows", [])
        if not batch:
            break
        rows.extend(batch)
        offset += len(batch)
        if offset >= data.get("rowCount", 0):
            break
    return pd.DataFrame(rows, columns=columns) if columns else pd.DataFrame(rows)

print(run_query("SELECT 1 AS ok"))

cols = run_query(f"""
    SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
    FROM INFORMATION_SCHEMA."COLUMNS"
    WHERE TABLE_SCHEMA = '{SCHEMA_PATH}'
""")
print(f"{cols['TABLE_NAME'].nunique()} tables, {len(cols)} columns in {SCHEMA_PATH}")

def tcols(t):
    return list(cols[cols["TABLE_NAME"].str.upper() == t.upper()]["COLUMN_NAME"])

def real(table, name):
    """Exact column name as stored in the table (case-insensitive lookup), or None."""
    return {c.upper(): c for c in tcols(table)}.get(name.upper())


   ok
0   1
110 tables, 6063 columns in publishFINDATAREP.IFCTRE.Operational_Data.Common


In [5]:
T_PNL, T_MV, T_TAD = "Inv_Pnl_Summary", "Trade_Mktval_Detail", "Trade_And_Asset_Detail"

PNL_COLUMNS = [
    "Business_Date", "MX_L3_PORTFO", "MX_L4_PORTFO", "Currency", "MX_CD_ELIG", "Trade_Typology",
    "Instrument", "Contract_Origin", "Trade_ID", "Country", "CTP_Code", "MX_DEAL_TYPE",
    "DTD_pnl_usd", "MTD_Pnl_USD", "YTD_pnl_USD", "LTD_pnl_usd", "Cash", "Cash_USD", "Market_Value_USD",
]
RENAME = {
    "MX_L3_PORTFO": "L3_PORTFO", "MX_L4_PORTFO": "L4_PORTFO", "Currency": "CURRENCY",
    "MX_CD_ELIG": "CD_ELIG", "Trade_Typology": "TYPOLOGY", "Instrument": "INSTRUMEN",
    "Contract_Origin": "CNT_ORG", "Trade_ID": "NB", "Country": "COUNTRY", "CTP_Code": "CTP",
    "MX_DEAL_TYPE": "DEAL_TYPE", "DTD_pnl_usd": "DTD_PL_US", "MTD_Pnl_USD": "MTD_PL_US",
    "YTD_pnl_USD": "YTD_PL_US", "LTD_pnl_usd": "LTD_PL_US", "Cash": "CASH",
    "Cash_USD": "CASH_US", "Market_Value_USD": "MV_US",
}

# Source column -> internal name
MV_WANT = {"Initial_Notional": "MV_INITIAL", "Current_Notional": "MV_CURRENT",
           "Notional_Currency": "MV_CCY", "Maturity_Date": "MV_MATURITY"}
TAD_WANT = {"Trade_Date": "TAD_TRADE_DATE", "Settlement_Date": "TAD_SETTLEMENT",
            "Maturity_Date": "TAD_MATURITY", "Initial_Notional": "TAD_INITIAL",
            "Current_Notional": "TAD_CURRENT", "Notional_Currency": "TAD_CCY",
            "Security_Code": "TAD_SECURITY_CODE"}

def clean_key(s):
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

def pull_small(table, want):
    """Pull the trade id, the date column if any, and the wanted columns of one table."""
    id_col   = real(table, "Trade_ID")
    date_col = next((real(table, c) for c in ("Business_Date", "Eod_Date", "As_Of_Date", "Valuation_Date")
                     if real(table, c)), None)
    found    = {real(table, k): v for k, v in want.items() if real(table, k)}
    missing  = [k for k in want if not real(table, k)]
    print(f"{table}: id={id_col} | date={date_col} | missing columns: {missing or 'none'}")
    if not id_col:
        raise SystemExit(f"No Trade_ID column in {table}. Columns: {tcols(table)[:60]}")

    select = ", ".join(f'"{c}"' for c in dict.fromkeys([id_col] + ([date_col] if date_col else []) + list(found)))
    where  = f'WHERE "{date_col}" = DATE \'{BUSINESS_DATE}\'' if date_col else ""
    if not date_col:
        print(f"  Warning: no date column in {table}; it may hold several dates per trade.")
    d = run_query(f'SELECT {select} FROM {SCHEMA_PATH}."{table}" {where}')
    d = d.rename(columns={**found, id_col: "_ID"})
    for v in want.values():
        if v not in d.columns:
            d[v] = pd.NA
    d["_KEY"] = clean_key(d["_ID"])
    n_dup = d.duplicated("_KEY", keep=False).sum()
    print(f"  rows={len(d)} | rows sharing a trade id: {n_dup}" + (" (first row kept)" if n_dup else ""))
    return d.drop_duplicates("_KEY")[["_KEY"] + list(want.values())]

# --- Pull ---
pnl_select = ", ".join(f'"{c}"' for c in PNL_COLUMNS)
pnl = run_query(f'SELECT {pnl_select} FROM {SCHEMA_PATH}."{T_PNL}" '
                f'WHERE "Business_Date" = DATE \'{BUSINESS_DATE}\'')
pnl = pnl.rename(columns=RENAME)
pnl["_KEY"] = clean_key(pnl["NB"])
print(f"{T_PNL}: rows={len(pnl)} | unique trade ids={pnl['_KEY'].nunique()}")

mv  = pull_small(T_MV,  MV_WANT)
tad = pull_small(T_TAD, TAD_WANT)

# --- Join ---
out = pnl.merge(mv,  on="_KEY", how="left", indicator="_MV_MATCH")
out = out.merge(tad, on="_KEY", how="left", indicator="_TAD_MATCH")
print(f"\nRows after joins: {len(out)} (PNL had {len(pnl)})")
print("No match in MKTVAL:", (out["_MV_MATCH"] == "left_only").sum())
print("No match in TAD   :", (out["_TAD_MATCH"] == "left_only").sum())

# --- Types ---
for c in ["MV_INITIAL", "MV_CURRENT", "TAD_INITIAL", "TAD_CURRENT"]:
    out[c] = pd.to_numeric(out[c], errors="coerce")
for c in ["MV_MATURITY", "TAD_TRADE_DATE", "TAD_SETTLEMENT", "TAD_MATURITY"]:
    out[c] = pd.to_datetime(out[c], errors="coerce")

# --- Final fields: Trade_And_Asset_Detail first, Trade_Mktval_Detail as fallback ---
out["TRADE_DATE"]           = out["TAD_TRADE_DATE"]
out["SETTLEMENT_DATE"]      = out["TAD_SETTLEMENT"]
out["MATURITY_DATE"]        = out["TAD_MATURITY"].combine_first(out["MV_MATURITY"])
out["INITIAL_NOTIONAL"]     = out["TAD_INITIAL"].combine_first(out["MV_INITIAL"])
out["OUTSTANDING_NOTIONAL"] = out["TAD_CURRENT"].combine_first(out["MV_CURRENT"])
out["NOTIONAL_CCY"]         = out["TAD_CCY"].combine_first(out["MV_CCY"])

# Cross-checks between the two sources (reported on the Checks sheet only)
out["CHK_MATURITY_MATCH"] = ((out["MV_MATURITY"] == out["TAD_MATURITY"])
                             | out["MV_MATURITY"].isna() | out["TAD_MATURITY"].isna())
out["CHK_NOTIONAL_DIFF"]  = out["MV_CURRENT"] - out["TAD_CURRENT"]

# --- Fill and zero counts ---
final_cols = ["TRADE_DATE", "SETTLEMENT_DATE", "MATURITY_DATE",
              "INITIAL_NOTIONAL", "OUTSTANDING_NOTIONAL", "NOTIONAL_CCY"]
print("\nFilled values per column (out of", len(out), "rows):")
print(out[final_cols].notna().sum().to_string())
print("\nInitial notional = 0     :", (out["INITIAL_NOTIONAL"] == 0).sum())
print("Outstanding notional = 0 :", (out["OUTSTANDING_NOTIONAL"] == 0).sum())
print("Trade_Mktval_Detail current notional = 0 :", (out["MV_CURRENT"] == 0).sum(),
      "(Trade_And_Asset_Detail:", (out["TAD_CURRENT"] == 0).sum(), ")")
display(out[["NB", "INSTRUMEN", "INITIAL_NOTIONAL", "OUTSTANDING_NOTIONAL", "NOTIONAL_CCY"]].head(10))


Inv_Pnl_Summary: rows=2324 | unique trade ids=2285
Trade_Mktval_Detail: id=Trade_ID | date=Business_Date | missing columns: none
  rows=52183 | rows sharing a trade id: 34436 (first row kept)
Trade_And_Asset_Detail: id=Trade_ID | date=Business_Date | missing columns: none
  rows=49310 | rows sharing a trade id: 29948 (first row kept)

Rows after joins: 2324 (PNL had 2324)
No match in MKTVAL: 0
No match in TAD   : 0

Filled values per column (out of 2324 rows):
TRADE_DATE              2324
SETTLEMENT_DATE         1731
MATURITY_DATE           2131
INITIAL_NOTIONAL        2324
OUTSTANDING_NOTIONAL    2324
NOTIONAL_CCY            2324

Initial notional = 0     : 71
Outstanding notional = 0 : 699
Trade_Mktval_Detail current notional = 0 : 2324 (Trade_And_Asset_Detail: 699 )


,NB,INSTRUMEN,INITIAL_NOTIONAL,OUTSTANDING_NOTIONAL,NOTIONAL_CCY
0,1734855,USD FIX ZC,2.000000e+08,2.000000e+08,USD
1,1734793,EUR FIX ZC,3.000000e+07,3.000000e+07,EUR
2,1734796,JPY FIX 6M,7.500000e+09,7.500000e+09,JPY
3,1734877,JPY FIX 6M,2.000000e+09,0.000000e+00,JPY
4,1734840,CHF FIX 6M,1.500000e+08,1.500000e+08,CHF
5,1734862,CHF FIX 6M,1.230000e+08,0.000000e+00,CHF
6,1734833,CHF FIX 6M,3.300000e+07,0.000000e+00,CHF
7,1734839,CHF FIX 6M,7.000000e+07,7.000000e+07,CHF
8,1734828,CHF FIX 6M,1.000000e+08,1.000000e+08,CHF
9,1734819,CHF FIX 6M,4.100000e+07,4.100000e+07,CHF


In [6]:
T_SEC = "Securities_Detail"
out = out.drop(columns=[c for c in ["ISIN", "ISIN_SRC", "_SEC_MATCH", "_SEC_KEY"] if c in out.columns])

sec_code = real(T_SEC, "Security_Code")
sec_isin = real(T_SEC, "ISIN")
print(f"Securities_Detail -> Security_Code={sec_code} | ISIN={sec_isin}")

out["_SEC_KEY"] = (out["TAD_SECURITY_CODE"].astype(str).str.strip()
                   .replace({"None": pd.NA, "nan": pd.NA, "<NA>": pd.NA}))
print("Trades carrying a security code:", out["_SEC_KEY"].notna().sum(), "of", len(out))

if sec_code and sec_isin:
    sec = run_query(f'SELECT DISTINCT "{sec_code}", "{sec_isin}" FROM {SCHEMA_PATH}."{T_SEC}"')
    sec.columns = ["_SEC_KEY", "ISIN_SRC"]
    sec["_SEC_KEY"] = sec["_SEC_KEY"].astype(str).str.strip()
    print(f"Securities_Detail: {len(sec)} rows | ISIN filled: {sec['ISIN_SRC'].notna().sum()}")
    sec = sec.drop_duplicates("_SEC_KEY")

    n = len(out)
    out = out.merge(sec, on="_SEC_KEY", how="left", indicator="_SEC_MATCH")
    out["ISIN"] = out["ISIN_SRC"]
    print(f"Rows: {n} -> {len(out)}")
    has_code = out["_SEC_KEY"].notna()
    print("Trades with a code that found a match :", ((out["_SEC_MATCH"] == "both") & has_code).sum())
    print("Trades with a code but no match       :", ((out["_SEC_MATCH"] == "left_only") & has_code).sum())
    print("Rows with an ISIN                     :", out["ISIN"].notna().sum())
else:
    out["ISIN"] = pd.NA
    out["_SEC_MATCH"] = "not_checked"
    print("ISIN left blank: Security_Code or ISIN column not found in Securities_Detail.")


Securities_Detail -> Security_Code=Security_Code | ISIN=ISIN
Trades carrying a security code: 791 of 2324
Securities_Detail: 10295 rows | ISIN filled: 9119
Rows: 2324 -> 2324
Trades with a code that found a match : 791
Trades with a code but no match       : 0
Rows with an ISIN                     : 791


In [7]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

layout = ["Business_Date", "L3_PORTFO", "L4_PORTFO", "CURRENCY", "CD_ELIG", "TYPOLOGY", "INSTRUMEN",
          "CNT_ORG", "NB", "COUNTRY", "CTP", "DEAL_TYPE", "DTD_PL_US", "MTD_PL_US", "YTD_PL_US",
          "LTD_PL_US", "CASH", "CASH_US", "MV_US",
          "ISIN", "TRADE_DATE", "SETTLEMENT_DATE", "MATURITY_DATE",
          "INITIAL_NOTIONAL", "OUTSTANDING_NOTIONAL", "NOTIONAL_CCY"]
NUMERIC_COLS = ["DTD_PL_US", "MTD_PL_US", "YTD_PL_US", "LTD_PL_US", "CASH", "CASH_US", "MV_US",
                "INITIAL_NOTIONAL", "OUTSTANDING_NOTIONAL"]
DATE_COLS = ["Business_Date", "TRADE_DATE", "SETTLEMENT_DATE", "MATURITY_DATE"]

df = out[layout].copy()
for c in NUMERIC_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], errors="coerce")

if L3_FILTER:
    keep = out["L3_PORTFO"].isin(L3_FILTER).values
    df_filtered, out_filtered = df[keep].reset_index(drop=True), out[keep].reset_index(drop=True)
else:
    df_filtered, out_filtered = df.copy(), out.copy()
print(f"Rows before filter: {len(df)} | after filter: {len(df_filtered)}")

filename = f"MX_Subaccount_PL_Extended_{BUSINESS_DATE.replace('-', '')}.xlsx"
os.makedirs(OUTPUT_DIR, exist_ok=True)
filepath = os.path.join(OUTPUT_DIR, filename)

has_code = out_filtered["_SEC_KEY"].notna()
checks_df = pd.DataFrame({
    "Check": [
        "Business date", "Rows exported (after L3 filter)", "Distinct trades exported",
        "Trades with no match in Trade_Mktval_Detail",
        "Trades with no match in Trade_And_Asset_Detail",
        "Rows carrying a security code",
        "Rows with a security code but no match in Securities_Detail",
        "Maturity date differs between the two tables",
        "Outstanding notional differs between the two tables (> 0.01)",
        "Rows where Trade_Mktval_Detail current notional = 0",
        "Rows where Trade_And_Asset_Detail current notional = 0",
        "Rows with no ISIN", "Rows with no trade date", "Rows with no settlement date",
        "Rows with no maturity date", "Rows with no initial notional",
        "Rows with no outstanding notional", "Extract generated on",
    ],
    "Value": [
        BUSINESS_DATE, len(df_filtered), int(df_filtered["NB"].nunique()),
        int((out_filtered["_MV_MATCH"] == "left_only").sum()),
        int((out_filtered["_TAD_MATCH"] == "left_only").sum()),
        int(has_code.sum()),
        int(((out_filtered["_SEC_MATCH"] == "left_only") & has_code).sum()),
        int((~out_filtered["CHK_MATURITY_MATCH"]).sum()),
        int((out_filtered["CHK_NOTIONAL_DIFF"].abs() > 0.01).sum()),
        int((out_filtered["MV_CURRENT"] == 0).sum()),
        int((out_filtered["TAD_CURRENT"] == 0).sum()),
        int(df_filtered["ISIN"].isna().sum()), int(df_filtered["TRADE_DATE"].isna().sum()),
        int(df_filtered["SETTLEMENT_DATE"].isna().sum()), int(df_filtered["MATURITY_DATE"].isna().sum()),
        int(df_filtered["INITIAL_NOTIONAL"].isna().sum()),
        int(df_filtered["OUTSTANDING_NOTIONAL"].isna().sum()),
        datetime.now().strftime("%Y-%m-%d %H:%M"),
    ],
})

with pd.ExcelWriter(filepath, engine="openpyxl",
                    datetime_format="yyyy-mm-dd", date_format="yyyy-mm-dd") as xw:
    df_filtered.to_excel(xw, index=False, sheet_name="PL_Summary")
    checks_df.to_excel(xw, index=False, sheet_name="Checks")

wb = load_workbook(filepath)
header_font  = Font(name="Calibri", bold=True, color="FFFFFF", size=11)
header_fill  = PatternFill(fill_type="solid", fgColor="003366")   # dark blue: original columns
new_fill     = PatternFill(fill_type="solid", fgColor="1F7A4D")   # green: added fields
header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
body_font    = Font(name="Calibri", size=10)
thin         = Side(style="thin", color="CCCCCC")
border       = Border(left=thin, right=thin, top=thin, bottom=thin)
band_fill    = PatternFill(fill_type="solid", fgColor="EEF2F7")
NEW_FIELDS   = {"ISIN", "TRADE_DATE", "SETTLEMENT_DATE", "MATURITY_DATE",
                "INITIAL_NOTIONAL", "OUTSTANDING_NOTIONAL", "NOTIONAL_CCY"}

def style_sheet(ws, numeric_cols=(), date_cols=()):
    for cell in ws[1]:
        cell.font, cell.alignment, cell.border = header_font, header_align, border
        cell.fill = new_fill if str(cell.value) in NEW_FIELDS else header_fill
    ws.row_dimensions[1].height = 30
    names = {c.column: str(c.value) for c in ws[1]}
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            n = names[cell.column]
            cell.font, cell.border = body_font, border
            if n in numeric_cols:
                cell.number_format = "#,##0.00"
                cell.alignment = Alignment(horizontal="right", vertical="center")
            elif n in date_cols:
                cell.number_format = "yyyy-mm-dd"
                cell.alignment = Alignment(horizontal="center", vertical="center")
            else:
                cell.alignment = Alignment(horizontal="left", vertical="center")
        if row[0].row % 2 == 0:
            for cell in row:
                cell.fill = band_fill
    for c in ws.columns:
        longest = max((len(str(x.value)) if x.value is not None else 0) for x in c[:200])
        ws.column_dimensions[get_column_letter(c[0].column)].width = min(longest + 4, 40)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

style_sheet(wb["PL_Summary"], numeric_cols=set(NUMERIC_COLS), date_cols=set(DATE_COLS))
style_sheet(wb["Checks"])
wb["Checks"].column_dimensions["A"].width = 62
wb["Checks"].column_dimensions["B"].width = 22
wb["Checks"].auto_filter.ref = None
wb.save(filepath)
print(f"File saved: {filepath}")


Rows before filter: 2324 | after filter: 2324
File saved: C:\Users\mkeita9\OneDrive - WBG\1. Work Documents\14. Audits\2026\MX_Subaccount_PL_Extended_20260831.xlsx


In [8]:
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print(f"Business date : {BUSINESS_DATE}")
print(f"Rows          : {len(df_filtered):,}  ({df_filtered['NB'].nunique():,} distinct trades)")
print(f"Columns       : {df_filtered.shape[1]}")
print(f"Saved file    : {filepath}\n")

print("=== Filled values in the added columns ===")
display(df_filtered[["ISIN", "TRADE_DATE", "SETTLEMENT_DATE", "MATURITY_DATE", "INITIAL_NOTIONAL",
                     "OUTSTANDING_NOTIONAL", "NOTIONAL_CCY"]].notna().sum().to_frame("Filled").T)

print("\n=== PL_Summary (first 50 rows) ===")
display(df_filtered.head(50))

print("\n=== Checks ===")
display(checks_df)

num_cols = ["DTD_PL_US", "MTD_PL_US", "YTD_PL_US", "LTD_PL_US", "CASH_US", "MV_US"]
print("\n=== Totals by L3 portfolio ===")
display(df_filtered.groupby("L3_PORTFO", dropna=False)[num_cols].sum().sort_values("MV_US", ascending=False))
print("\n=== Grand totals ===")
display(df_filtered[num_cols].sum().to_frame("Total").T)


Business date : 2026-08-31
Rows          : 2,324  (2,285 distinct trades)
Columns       : 26
Saved file    : C:\Users\mkeita9\OneDrive - WBG\1. Work Documents\14. Audits\2026\MX_Subaccount_PL_Extended_20260831.xlsx

=== Filled values in the added columns ===


,ISIN,TRADE_DATE,SETTLEMENT_DATE,MATURITY_DATE,INITIAL_NOTIONAL,OUTSTANDING_NOTIONAL,NOTIONAL_CCY
Filled,791,2324,1731,2131,2324,2324,2324



=== PL_Summary (first 50 rows) ===


,Business_Date,L3_PORTFO,L4_PORTFO,CURRENCY,CD_ELIG,TYPOLOGY,INSTRUMEN,CNT_ORG,NB,COUNTRY,CTP,DEAL_TYPE,DTD_PL_US,MTD_PL_US,YTD_PL_US,LTD_PL_US,CASH,CASH_US,MV_US,ISIN,TRADE_DATE,SETTLEMENT_DATE,MATURITY_DATE,INITIAL_NOTIONAL,OUTSTANDING_NOTIONAL,NOTIONAL_CCY
0,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,USD,Y,Loan Deposit,USD FIX ZC,1734256,1734855,SG,ROYCSP,Asset,"60,896.84","539,828.78","539,828.78","539,828.78","-200,000,000.00","-200,000,000.00","200,539,828.78",NaN,2026-08-04,2026-08-04,2026-09-10,"200,000,000.00","200,000,000.00",USD
1,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,EUR,Y,Loan Deposit,EUR FIX ZC,1734194,1734793,NL,KOEXNL,Asset,"6,569.18","57,846.27","57,846.27","57,846.27","-34,846,710.00","-34,846,710.00","34,904,556.27",NaN,2026-08-05,2026-08-05,2026-09-08,"30,000,000.00","30,000,000.00",EUR
2,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,JPY,Y,Loan Deposit,JPY FIX 6M,1734197,1734796,FR,SGIZFR,Asset,"5,107.94","26,252.36","26,252.36","26,252.36","-46,941,319.28","-46,941,319.28","46,967,571.64",NaN,2026-08-11,2026-08-11,2026-09-15,"7,500,000,000.00","7,500,000,000.00",JPY
3,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,JPY,Y,Loan Deposit,JPY FIX 6M,1734278,1734877,SG,STBCSG,Asset,23.73,"10,890.39","10,890.39","10,890.39","10,890.39","10,890.39",0.00,NaN,2026-07-21,2026-07-21,2026-08-21,"2,000,000,000.00",0.00,JPY
4,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,CHF,Y,Loan Deposit,CHF FIX 6M,1734241,1734840,CH,BCGECH,Asset,888.26,"6,069.18","6,069.18","6,069.18","-185,534,333.31","-185,534,333.31","185,540,402.49",NaN,2026-08-10,2026-08-10,2026-09-15,"150,000,000.00","150,000,000.00",CHF
5,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,CHF,Y,Loan Deposit,CHF FIX 6M,1734263,1734862,CH,BCVLLA,Asset,-7.90,"-6,339.09","-6,339.09","-6,339.09","-6,339.09","-6,339.09",0.00,NaN,2026-07-24,2026-07-24,2026-08-28,"123,000,000.00",0.00,CHF
6,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,CHF,Y,Loan Deposit,CHF FIX 6M,1734234,1734833,CH,BLKBLI,Asset,-0.99,-793.68,-793.68,-793.68,-793.68,-793.68,0.00,NaN,2026-08-13,2026-08-13,2026-08-27,"33,000,000.00",0.00,CHF
7,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,CHF,Y,Loan Deposit,CHF FIX 6M,1734240,1734839,CH,BLKBLI,Asset,379.03,218.11,218.11,218.11,"-86,582,688.88","-86,582,688.88","86,582,906.99",NaN,2026-08-06,2026-08-06,2026-09-11,"70,000,000.00","70,000,000.00",CHF
8,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,CHF,Y,Loan Deposit,CHF FIX 6M,1734229,1734828,CH,KBTGCH,Asset,593.68,"5,248.78","5,248.78","5,248.78","-123,689,555.54","-123,689,555.54","123,694,804.32",NaN,2026-08-10,2026-08-10,2026-09-15,"100,000,000.00","100,000,000.00",CHF
9,2026-08-31,LAM_LIQUIDTY,LAM_LIQUIDITY,CHF,Y,Loan Deposit,CHF FIX 6M,1734220,1734819,CH,LUKBLU,Asset,238.40,"2,078.68","2,078.68","2,078.68","-50,712,717.77","-50,712,717.77","50,714,796.45",NaN,2026-08-12,2026-08-12,2026-09-16,"41,000,000.00","41,000,000.00",CHF



=== Checks ===


,Check,Value
0,Business date,2026-08-31
1,Rows exported (after L3 filter),2324
2,Distinct trades exported,2285
3,Trades with no match in Trade_Mktval_Detail,0
4,Trades with no match in Trade_And_Asset_Detail,0
5,Rows carrying a security code,791
6,Rows with a security code but no match in Secu...,0
7,Maturity date differs between the two tables,0
8,Outstanding notional differs between the two t...,1625
9,Rows where Trade_Mktval_Detail current notiona...,2324



=== Totals by L3 portfolio ===


,DTD_PL_US,MTD_PL_US,YTD_PL_US,LTD_PL_US,CASH_US,MV_US
L3_PORTFO,,,,,,
LAM_STRATEGIC,"41,377,309.43","18,197,224,922.53","18,197,224,922.53","18,197,224,922.53","-192,616,019.30","18,389,840,942.01"
LAM,"-241,232,072.47","17,696,136,754.29","17,696,136,754.29","17,696,136,754.29","-53,468,858.36","17,749,605,612.66"
LAM_LIQUIDTY,"-7,511,393.46","7,945,523,753.69","7,945,523,753.69","7,945,523,753.69","270,116,440.37","7,675,407,313.34"
LCY_POOLS,"-125,332.58","585,005,030.34","585,005,030.34","585,005,030.34","22,022,163.51","562,982,866.81"
OPER_CASH1,"41,414.30","-16,105.43","-16,105.43","-16,105.43","-280,573,969.27","280,557,863.82"
LEGACY_INR,"2,416.56","4,987,708.91","4,987,708.91","4,987,708.91","2,079.39","4,985,629.51"
MF_MOPS_CASH,"251,117,408.91","472,668,089.71","472,668,089.71","472,668,089.71","472,672,937.25","-4,847.54"
IFC_DIN,"-761,693.84","-16,220,909.24","-16,220,909.24","-16,220,909.24","2,401,506,248.19","-2,417,727,157.43"



=== Grand totals ===


,DTD_PL_US,MTD_PL_US,YTD_PL_US,LTD_PL_US,CASH_US,MV_US
Total,"42,908,056.85","44,885,309,244.80","44,885,309,244.80","44,885,309,244.80","2,639,661,021.79","42,245,648,223.17"
